In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")

    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth

!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Phi-4-unsloth-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.39G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/1.03G [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj"
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

Unsloth 2026.3.4 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


In [4]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="phi-4",
)

In [5]:
!pip install datasets -q

In [16]:
from datasets import load_dataset
import re
dataset = load_dataset("Jaiccc/model0_boundary_predict_streaming", split="train")

**Display a particular sample's input context and model's prediction on that sample**

In [30]:
sample_number = 27
sample = dataset[sample_number]
print("Dataset columns available:", sample.keys())
instruction = sample['instruction']
input_data = sample['input']
expected_output = sample['output']

# Combine the instruction and input into a single user message
combined_user_text = f"{instruction}\n\n{input_data}"

# Format the prompt to match the ChatML-style markers
prompt = f"<|im_start|>user<|im_sep|>{combined_user_text}<|im_end|><|im_start|>assistant<|im_sep|>"
inputs = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")
outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=128,
    use_cache=True,
    temperature=0.2,
)

# Decode the raw output
raw_output = tokenizer.batch_decode(outputs, clean_up_tokenization_spaces=True)[0]

# Extract the assistant block using the exact marker tokens
m = re.search(r'<\|im_start\|>assistant<\|im_sep\|>(.*?)<\|im_end\|>', raw_output, re.S)
raw_assistant = m.group(1).strip()
print(f"=== ROW {sample_number} COMBINED INPUT ===")
print(combined_user_text)
print("\n=== EXPECTED OUTPUT (From Dataset) ===")
print(expected_output)
print("\n=== MODEL GENERATED OUTPUT ===")
print(raw_assistant)

Dataset columns available: dict_keys(['instruction', 'input', 'output'])
=== ROW 27 COMBINED INPUT ===
Your task is to analyze terminal XML logs and determine whether the timestamp in the TARGET LINE belongs to a "new event" or an "old event".

### DEFINITION OF A NEW EVENT:
1. **Explicit Prompts:** The very first `<user_input>` that immediately follows a shell prompt (e.g., `demo@faiserver:~$`).
2. **Phase Transitions:** In automated logs, moving from one major build stage to another (e.g., from 'fai-mirror finished' to 'Copying the nfsroot').
3. **Internal Logic:** Shifts from downloading to processing.

### WHAT IS *NOT* A NEW EVENT (OLD EVENT):
- **User Input / Keystrokes:** A user typing a command, including pressing the Enter key (a newline `\n`), is just the completion of the input phase.
- **Incomplete Tasks:** Continuous system output without a clear phase shift.

CRITICAL INSTRUCTION: You must classify ONLY the timestamp found in the "### TARGET LINE" section. Do NOT extract 

**Performs a comprehensive evaluation by iterating through every sample in the dataset to validate the model's predictions against the predefined ground truth. For each entry, it compares the assistant's classification to the expected label and calculates a final accuracy metric to quantify the model's overall performance.**

In [26]:
# 1. Initialize counters for a quick summary
total_samples = len(dataset)
correct_count = 0

print(f"Scanning {total_samples} samples...\n")

for i in range(total_samples):
    sample = dataset[i]

    # 2. Extract components
    instruction = sample['instruction']
    input_data = sample['input']
    expected_output = sample['output'].strip()

    # 3. Format the prompt
    combined_user_text = f"{instruction}\n\n{input_data}"
    prompt = f"<|im_start|>user<|im_sep|>{combined_user_text}<|im_end|><|im_start|>assistant<|im_sep|>"

    # 4. Generate
    inputs = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=64, # Classification is short, 64 is plenty
        use_cache=True,
        temperature=0.1,   # Lower temperature for more consistent classification
    )

    # 5. Decode and Clean
    raw_output = tokenizer.batch_decode(outputs, clean_up_tokenization_spaces=True)[0]

    # Extract only the assistant's specific response
    m = re.search(r'<\|im_start\|>assistant<\|im_sep\|>(.*?)<\|im_end\|>', raw_output, re.S)
    model_result = m.group(1).strip() if m else raw_output.split("assistant")[-1].strip()

    # 6. Compare and Print
    is_correct = "✅" if model_result == expected_output else "❌"
    if model_result == expected_output:
        correct_count += 1

    print(f"--- Sample {i} ---")
    print(f"EXPECTED: {expected_output}")
    print(f"MODEL:    {model_result} {is_correct}")
    print("-" * 20)

# 7. Final Summary
accuracy = (correct_count / total_samples) * 100
print(f"\nFinal Accuracy: {accuracy:.2f}% ({correct_count}/{total_samples})")

Scanning 30 samples...

--- Sample 0 ---
EXPECTED: 5.938425, old event
MODEL:    5.938425, old event ✅
--------------------
--- Sample 1 ---
EXPECTED: 13.357503, old event
MODEL:    13.357503, old event ✅
--------------------
--- Sample 2 ---
EXPECTED: 21.987222, new event
MODEL:    21.987222, new event ✅
--------------------
--- Sample 3 ---
EXPECTED: 22.208425, old event
MODEL:    22.208425, old event ✅
--------------------
--- Sample 4 ---
EXPECTED: 37.178971, new event
MODEL:    37.178971, old event ❌
--------------------
--- Sample 5 ---
EXPECTED: 37.948668, old event
MODEL:    37.948668, old event ✅
--------------------
--- Sample 6 ---
EXPECTED: 41.941309, new event
MODEL:    41.941309, new event ✅
--------------------
--- Sample 7 ---
EXPECTED: 44.147483, new event
MODEL:    44.147483, new event ✅
--------------------
--- Sample 8 ---
EXPECTED: 45.646049, old event
MODEL:    45.646049, old event ✅
--------------------
--- Sample 9 ---
EXPECTED: 50.572229, new event
MODEL:    50